In [11]:
import igl
import numpy as np
import scipy.sparse as sp
import meshplot as mp

V, F = igl.read_triangle_mesh("gaudi.ply")

In [12]:
G = igl.grad(V, F)
f = V[:, 2]  # scalar function on vertices
grad_f = G @ f

p = mp.plot(V, F, c=f, shading={"wireframe": False})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

In [13]:
# gradient operator
G = igl.grad(V, F)

# per-face gradients
g = G @ f
grad_f = np.asarray(g).reshape(-1, 3)

# face barycenters
BC = igl.barycenter(V, F)

# normalize for display
grad_norm = np.linalg.norm(grad_f, axis=1, keepdims=True)
grad_unit = grad_f / np.maximum(grad_norm, 1e-12)

# visualize
p = mp.plot(V, F, c=f)
scale = 1
p.add_lines(BC, BC + scale * grad_unit)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

1

In [22]:
from meshplot import plot, subplot, interact

g = igl.grad(V,F)
gf = g.dot(f).reshape(F.shape, order="F")

gf_mag = np.linalg.norm(gf, axis=1)
p = plot(V,F, f, shading={"wireframe":False}, return_plot=True)

max_size = igl.avg_edge_length(V,F) / np.mean(gu_mag)
bc = igl.barycenter(V,F)
bcn = bc + max_size * gf
p.add_lines(bc, bcn, shading={"line_color": "black"})

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

1

In [23]:
p = plot(V,F, gf_mag, shading={"wireframe":False}, return_plot=True)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

In [31]:
# area-normalized curvature
M = igl.massmatrix(V, F, igl.MASSMATRIX_TYPE_VORONOI)

K = np.asarray(igl.gaussian_curvature(V, F)).reshape(-1) / mass

# tighter symmetric range
m = np.percentile(np.abs(K), 95)

mp.plot(
    V, F,
    c=K,
    shading={
        "colormap": "RdBu",
        "vmin": -m,
        "vmax":  m
    }
)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

In [ ]:
from scipy.sparse.linalg import spsolve

l = igl.cotmatrix(V, F)
m = igl.massmatrix(V, F, igl.MASSMATRIX_TYPE_VORONOI)

mass = np.maximum(m.diagonal(), 1e-12)
minv = sp.diags(1.0 / mass)


# unsigned mean curvature from Laplacian
hn = -minv.dot(l.dot(V))
h = np.linalg.norm(hn, axis=1)
plot(V, F, h)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

In [ ]:
v1, v2, k1, k2 = igl.principal_curvature(V, F)

# signed mean curvature from principal curvatures
h2 = 0.5 * (k1 + k2)
p = plot(V, F, h2, shading={"wireframe": False}, return_plot=True)
print(np.min(h2), np.max(h2))
avg = igl.avg_edge_length(V, F) / 2.0
p.add_lines(V + v1 * avg, V - v1 * avg, shading={"line_color": "red"})
p.add_lines(V + v2 * avg, V - v2 * avg, shading={"line_color": "green"})


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(-33.71693…

-0.10902205866814077 0.22713132197935365


2